# 🔍 RAG-based PDF Question Answering System

**Tech Stack:** LangChain · FAISS · SentenceTransformers · HuggingFace · Gradio

**Pipeline:**
```
PDF Upload → Text Chunking → Embeddings → FAISS Index → Semantic Retrieval → LLM Answer
```

> ⚡ Make sure you have **GPU enabled**: Runtime → Change runtime type → T4 GPU

## Cell 1 — Install Dependencies

In [ ]:
# Install all required libraries
!pip install -q langchain langchain-community faiss-cpu sentence-transformers
!pip install -q pypdf transformers accelerate gradio
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118

print("✅ All dependencies installed!")

## Cell 2 — Imports & Config

In [ ]:
import os
import torch
import numpy as np
from pathlib import Path

# LangChain
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate

# HuggingFace
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# Config
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 3
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "google/flan-t5-base"  # Lightweight, runs on free Colab GPU

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Cell 3 — Upload & Load PDFs

In [ ]:
from google.colab import files

# Upload one or more PDFs
print("📂 Upload your PDF file(s):")
uploaded = files.upload()

all_documents = []

for filename in uploaded.keys():
    print(f"\n📄 Loading: {filename}")
    loader = PyPDFLoader(filename)
    pages = loader.load()
    all_documents.extend(pages)
    print(f"   → {len(pages)} pages loaded")

print(f"\n✅ Total pages across all PDFs: {len(all_documents)}")

## Cell 4 — Chunk Text

In [ ]:
# Split documents into overlapping chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_documents(all_documents)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"\n📝 Sample chunk:")
print("-" * 60)
print(chunks[0].page_content[:300])
print("-" * 60)

## Cell 5 — Generate Embeddings & Build FAISS Index

In [ ]:
import time

print(f"⚙️  Loading embedding model: {EMBED_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}  # Cosine similarity ready
)

print("🔢 Generating embeddings and building FAISS index...")
start = time.time()
vectorstore = FAISS.from_documents(chunks, embeddings)
elapsed = time.time() - start

# Save index locally
vectorstore.save_local("faiss_index")

print(f"✅ FAISS index built in {elapsed:.1f}s")
print(f"✅ Index saved to ./faiss_index")
print(f"✅ Total vectors indexed: {vectorstore.index.ntotal}")

## Cell 6 — Load LLM & Build RAG Chain

In [ ]:
print(f"⚙️  Loading LLM: {LLM_MODEL}")

# Using Flan-T5 (lightweight, free, works on Colab T4)
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)

pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device=0 if device == "cuda" else -1
)

llm = HuggingFacePipeline(pipeline=pipe)

# Custom RAG prompt
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Answer the question based only on the context below.\n"
        "If the answer is not in the context, say 'I don't know.'\n\n"
        "Context: {context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    )
)

# Build RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": TOP_K}),
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)

print("✅ RAG chain ready!")

## Cell 7 — Ask Questions

In [ ]:
def ask(question: str):
    """Query the RAG pipeline and return answer + sources."""
    result = qa_chain({"query": question})

    print(f"\n❓ Question: {question}")
    print(f"\n💡 Answer: {result['result']}")
    print(f"\n📚 Sources retrieved (top {TOP_K}):")
    for i, doc in enumerate(result["source_documents"]):
        page = doc.metadata.get("page", "?")
        print(f"  [{i+1}] Page {page}: {doc.page_content[:120]}...")
    return result

# Try it!
result = ask("What is the main topic of this document?")

## Cell 8 — Evaluate: Recall@K and MRR

In [ ]:
def recall_at_k(query: str, relevant_keywords: list, k: int = 3) -> float:
    """
    Estimate Recall@K by checking if retrieved chunks contain relevant keywords.
    relevant_keywords: list of strings that should appear in a good answer.
    """
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.get_relevant_documents(query)

    hits = 0
    for kw in relevant_keywords:
        if any(kw.lower() in doc.page_content.lower() for doc in docs):
            hits += 1

    return hits / len(relevant_keywords) if relevant_keywords else 0.0


def mean_reciprocal_rank(query: str, relevant_keyword: str, k: int = 5) -> float:
    """
    Compute MRR: rank of the first relevant result.
    """
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.get_relevant_documents(query)

    for rank, doc in enumerate(docs, start=1):
        if relevant_keyword.lower() in doc.page_content.lower():
            return 1.0 / rank
    return 0.0


# ---- Example evaluation (update with your own test cases) ----
test_cases = [
    {
        "query": "What is the main topic?",
        "keywords": ["introduction", "abstract", "overview"],
        "mrr_keyword": "introduction"
    },
    {
        "query": "What are the conclusions?",
        "keywords": ["conclusion", "result", "finding"],
        "mrr_keyword": "conclusion"
    },
]

recalls, mrrs = [], []
for tc in test_cases:
    r = recall_at_k(tc["query"], tc["keywords"], k=TOP_K)
    m = mean_reciprocal_rank(tc["query"], tc["mrr_keyword"])
    recalls.append(r)
    mrrs.append(m)
    print(f"Query: {tc['query'][:50]}")
    print(f"  Recall@{TOP_K}: {r:.2f}  |  MRR: {m:.2f}\n")

print(f"📊 Mean Recall@{TOP_K}: {np.mean(recalls):.2f}")
print(f"📊 Mean MRR:       {np.mean(mrrs):.2f}")

## Cell 9 — Gradio UI (Interactive Demo)

In [ ]:
import gradio as gr

def answer_question(question: str) -> str:
    if not question.strip():
        return "Please enter a question."
    result = qa_chain({"query": question})
    answer = result["result"]
    sources = result["source_documents"]

    source_text = "\n\n📚 Sources:\n"
    for i, doc in enumerate(sources):
        page = doc.metadata.get("page", "?")
        source_text += f"[{i+1}] Page {page}: {doc.page_content[:150]}...\n"

    return answer + source_text


demo = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(lines=2, placeholder="Ask a question about your PDF...", label="Your Question"),
    outputs=gr.Textbox(lines=10, label="Answer + Sources"),
    title="📄 RAG PDF Question Answering System",
    description="Upload a PDF and ask questions. Powered by FAISS + SentenceTransformers + Flan-T5.",
    examples=[
        ["What is the main topic of this document?"],
        ["What are the key findings or conclusions?"],
        ["Summarize the introduction."],
    ]
)

# share=True gives a public URL — screenshot this for your portfolio!
demo.launch(share=True)